# Implement Adam from Scratch - SOLUTION

**Difficulty**: 🔴 Hard

**Companies**: Meta, Google

---

### Problem Statement

"Write Adam from scratch" is a classic ML interview ask — it tests whether you know what an optimizer actually *does* between `loss.backward()` and `opt.step()`.

Adam — **Adaptive Moment Estimation** (Kingma & Ba, 2014) — combines two ideas:

- **Momentum** — an exponential moving average of past gradients (1st moment).
- **RMSProp** — a per-parameter learning rate from a moving average of squared gradients (2nd moment).

Bias correction compensates for the moments starting at zero during the first steps.

### Tasks

1. `MyAdam` — the Adam update rule, including bias correction, following PyTorch's `torch.optim.Optimizer` interface.

### References

- Adam paper: https://arxiv.org/abs/1412.6980

In [ ]:
import math
import torch
from torch.optim import Optimizer


## Part 1: Adam

Adam combines two ideas:

- **Momentum** — an exponential moving average of past gradients.
- **RMSProp** — a per-parameter learning rate from a moving average of squared gradients.

```
m_t = β₁·m_{t−1} + (1−β₁)·g_t        (biased first moment)
v_t = β₂·v_{t−1} + (1−β₂)·g_t²       (biased second moment)
m̂_t = m_t / (1 − β₁ᵗ)                (bias corrections)
v̂_t = v_t / (1 − β₂ᵗ)
θ_t = θ_{t−1} − lr·m̂_t / (√v̂_t + ε)
```

**Key detail:** `β₁ᵗ` is β₁ raised to the **step count**, and the counter starts at 1. Skipping the bias correction is the most common Adam bug — it makes the first few steps roughly 3× too large.


In [ ]:
class MyAdam(Optimizer):
    """
    Adam optimizer.

    Args:
        params: iterable of parameters to optimize
        lr:     learning rate (default 1e-3)
        betas:  coefficients for the 1st- and 2nd-moment estimates
                (default (0.9, 0.999))
        eps:    term added to the denominator for numerical stability
                (default 1e-8)
    """

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        if lr < 0.0:
            raise ValueError(f"Invalid learning rate: {lr}")
        if not 0.0 <= betas[0] < 1.0:
            raise ValueError(f"Invalid beta1: {betas[0]}")
        if not 0.0 <= betas[1] < 1.0:
            raise ValueError(f"Invalid beta2: {betas[1]}")
        defaults = dict(lr=lr, betas=betas, eps=eps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        """
        Perform a single optimization step.

        Args:
            closure: optional callable that re-evaluates the model and returns
                     the loss
        Returns:
            the loss from the closure, if one was provided
        """
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr = group['lr']
            beta1, beta2 = group['betas']
            eps = group['eps']

            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad

                if grad.is_sparse:
                    raise RuntimeError("MyAdam does not support sparse gradients")

                state = self.state[p]

                # State initialization
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)

                exp_avg: torch.Tensor = state['exp_avg']
                exp_avg_sq: torch.Tensor = state['exp_avg_sq']
                state['step'] += 1

                # Biased moment estimates
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                # Bias corrections
                bias_correction1 = 1 - beta1 ** state['step']
                bias_correction2 = 1 - beta2 ** state['step']

                # Update (formulated to match torch.optim.Adam exactly)
                denom = (exp_avg_sq.sqrt() / math.sqrt(bias_correction2)).add_(eps)
                step_size = lr / bias_correction1
                p.addcdiv_(exp_avg, denom, value=-step_size)

        return loss


## Validation

`MyAdam` is compared step-for-step against `torch.optim.Adam` on a quadratic bowl — they should match to float precision.

Until your implementation is in place this test will fail — that is expected.


In [ ]:
def test_adam():
    """Compare MyAdam with torch.optim.Adam on a quadratic objective."""
    print("Testing Adam...", end=" ")

    torch.manual_seed(42)
    D = 64
    target = torch.linspace(-1, 1, D)

    x_ref = torch.zeros(D, requires_grad=True)
    opt_ref = torch.optim.Adam([x_ref], lr=0.1, betas=(0.9, 0.999), eps=1e-8)

    x_my = torch.zeros(D, requires_grad=True)
    opt_my = MyAdam([x_my], lr=0.1, betas=(0.9, 0.999), eps=1e-8)

    for _ in range(100):
        loss_ref = ((x_ref - target) ** 2).mean()
        opt_ref.zero_grad()
        loss_ref.backward()
        opt_ref.step()

        loss_my = ((x_my - target) ** 2).mean()
        opt_my.zero_grad()
        loss_my.backward()
        opt_my.step()

    diff = (x_ref - x_my).abs().max().item()
    if diff < 1e-5:
        print(f"PASS  (max parameter diff: {diff:.2e})")
    else:
        print(f"FAIL  (max parameter diff: {diff:.2e})")


